# Ai2020 기반 DFN 모델 — 1단계 간결본

**목적:** 공통 물리 보정 후, 양극 OCP 교체와 stoichiometry window 재최적화의 효과만 검증한다.

1. 공통 기반: Ai2020 `D_e` 단위 보정 + 측정 `R_n/R_p`
2. 기준: Ai2020 음극/양극 OCP + Ai2020 공개 window
3. 개선: Ai2020 음극 OCP + 실험 양극 GITT OCP + window 재최적화
4. 검증: C/50 qOCV 및 0.5C/1C/2C 충·방전 전압·cut-off 용량

`D_s`, `k`, Bruggeman 계수, 전해질 parameter set 및 fitting은 이 노트북의 범위가 아니다.


## 0. 실행 환경과 입력 파일

아래 5개 원본 파일을 노트북과 같은 폴더에 둔다.

- `260610 저율데이터 1.xlsx`
- `260610 저율데이터 2.xlsx`
- `260808 Eneterch셀 열화데이터 v1 6-6.xlsx`
- `260816 Positive GITT 7-7.xlsx`
- `260816 Positive GITT 7-8.xlsx`

음극 OCP는 Ai2020을 사용하므로 음극 half-cell 파일은 필요하지 않다.


In [ ]:
# Colab 최초 실행 시 주석을 해제한다.
# PyBaMM 26.7.1의 Ai2020 D_e를 명시적으로 보정해 기존 분석을 재현한다.
# %pip install -q pybamm==26.7.1.0 openpyxl

from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pybamm
from IPython.display import display
from scipy.integrate import cumulative_trapezoid
from scipy.optimize import differential_evolution, minimize

warnings.filterwarnings("ignore", message="Workbook contains no default style")
REQUIRED_PYBAMM = "26.7.1.0"
if pybamm.__version__ != REQUIRED_PYBAMM:
    raise RuntimeError(
        f"PyBaMM {REQUIRED_PYBAMM}이 필요합니다. 현재 버전: {pybamm.__version__}"
    )

FULL_CELL_LOW_RATE_FILES = [
    "260610 저율데이터 1.xlsx",
    "260610 저율데이터 2.xlsx",
]
DYNAMIC_FILE = "260808 Eneterch셀 열화데이터 v1 6-6.xlsx"
CATHODE_GITT_FILES = [
    "260816 Positive GITT 7-7.xlsx",
    "260816 Positive GITT 7-8.xlsx",
]
required_files = FULL_CELL_LOW_RATE_FILES + [DYNAMIC_FILE] + CATHODE_GITT_FILES
missing_files = [name for name in required_files if not Path(name).exists()]
if missing_files:
    raise FileNotFoundError("업로드가 필요한 파일: " + ", ".join(missing_files))

print("PyBaMM:", pybamm.__version__)
print("입력 파일 확인:", len(required_files), "개")


In [ ]:
# [0-2] C/50 qOCV와 Cell 6-6의 0.5C/1C/2C 원본 데이터 불러오기
TEST_C_RATES = (0.5, 1.0, 2.0)
SOC_GRID = np.linspace(0.0, 1.0, 1001)
DYNAMIC_STEP_INDEX = {
    (0.5, True): 7,
    (0.5, False): 10,
    (1.0, True): 12,
    (1.0, False): 15,
    (2.0, True): 17,
    (2.0, False): 20,
}


def clean_curve(horizontal, vertical):
    horizontal = np.asarray(horizontal, dtype=float).reshape(-1)
    vertical = np.asarray(vertical, dtype=float).reshape(-1)
    valid = np.isfinite(horizontal) & np.isfinite(vertical)
    order = np.argsort(horizontal[valid], kind="stable")
    horizontal = horizontal[valid][order]
    vertical = vertical[valid][order]
    unique_horizontal, inverse = np.unique(horizontal, return_inverse=True)
    if len(unique_horizontal) == len(horizontal):
        return horizontal, vertical
    return unique_horizontal, np.bincount(inverse, weights=vertical) / np.bincount(inverse)


def mean_overlap(curves, number_of_points=3001):
    lower = max(curve[0].min() for curve in curves)
    upper = min(curve[0].max() for curve in curves)
    grid = np.linspace(lower, upper, number_of_points)
    values = np.vstack([np.interp(grid, *curve) for curve in curves])
    return grid, values.mean(axis=0)


def load_full_cell_low_rate(file_names):
    charge_curves, discharge_curves = [], []
    charge_capacities, discharge_capacities = [], []

    for file_name in file_names:
        data = pd.read_excel(file_name, sheet_name="record")
        contiguous_step = data["Step Type"].astype(str).ne(
            data["Step Type"].astype(str).shift()
        ).cumsum()
        branches = {"CC Chg": [], "CC DChg": []}

        for _, branch in data.groupby(contiguous_step):
            step_type = str(branch["Step Type"].iloc[0])
            if step_type not in branches:
                continue
            capacity = pd.to_numeric(branch["Capacity(Ah)"], errors="coerce").to_numpy(float)
            voltage = pd.to_numeric(branch["Voltage(V)"], errors="coerce").to_numpy(float)
            current = float(pd.to_numeric(branch["Current(A)"], errors="coerce").median())
            valid = np.isfinite(capacity) & np.isfinite(voltage)
            capacity, voltage = capacity[valid], voltage[valid]
            if len(capacity) < 50 or capacity.max() < 2.2 or abs(current) > 0.06:
                continue

            end_capacity = float(capacity.max())
            branch_soc = (
                capacity / end_capacity
                if step_type == "CC Chg"
                else 1.0 - capacity / end_capacity
            )
            branch_soc, voltage = clean_curve(branch_soc, voltage)
            branches[step_type].append(
                (end_capacity, np.interp(SOC_GRID, branch_soc, voltage))
            )

        # 첫 방전은 부분 초기충전 직후이므로 제외한다.
        for capacity, voltage in branches["CC Chg"]:
            charge_capacities.append(capacity)
            charge_curves.append(voltage)
        for capacity, voltage in branches["CC DChg"][1:]:
            discharge_capacities.append(capacity)
            discharge_curves.append(voltage)

    mean_charge = np.mean(charge_curves, axis=0)
    mean_discharge = np.mean(discharge_curves, axis=0)
    return {
        "charge": mean_charge,
        "discharge": mean_discharge,
        "qocv": 0.5 * (mean_charge + mean_discharge),
        "capacity_ah": float(
            0.5 * (np.mean(charge_capacities) + np.mean(discharge_capacities))
        ),
        "number_of_charge_curves": len(charge_curves),
        "number_of_discharge_curves": len(discharge_curves),
    }


def load_dynamic_data(file_name):
    step = pd.read_excel(file_name, sheet_name="step")
    record = pd.read_excel(file_name, sheet_name="record")
    record["Date"] = pd.to_datetime(record["Date"])
    experiment = {}

    for key, step_index in DYNAMIC_STEP_INDEX.items():
        branch_row = step[
            (step["Cycle Index"] == 1) & (step["Step Index"] == step_index)
        ].iloc[0]
        previous_row = step[
            (step["Cycle Index"] == 1)
            & (step["Step Number"] == int(branch_row["Step Number"]) - 1)
        ].iloc[0]

        branch = record[
            (record["Date"] >= pd.to_datetime(branch_row["Oneset Date"]))
            & (record["Date"] <= pd.to_datetime(branch_row["End Date"]))
            & (record["Step Type"].astype(str) == str(branch_row["Step Type"]))
        ].copy()
        previous_rest = record[
            (record["Date"] >= pd.to_datetime(previous_row["Oneset Date"]))
            & (record["Date"] <= pd.to_datetime(previous_row["End Date"]))
            & (record["Step Type"].astype(str) == "Rest")
        ].copy()

        time_min = (
            branch["Date"] - branch["Date"].iloc[0]
        ).dt.total_seconds().to_numpy(float) / 60.0
        capacity = pd.to_numeric(branch["Capacity(Ah)"], errors="coerce").to_numpy(float)
        capacity = capacity - capacity[0]
        experiment[key] = {
            "time_min": time_min,
            "voltage_v": pd.to_numeric(branch["Voltage(V)"], errors="coerce").to_numpy(float),
            "capacity_ah": capacity,
            "current_a": abs(float(pd.to_numeric(branch["Current(A)"], errors="coerce").median())),
            "rest_voltage_v": float(pd.to_numeric(previous_rest["Voltage(V)"], errors="coerce").iloc[-1]),
        }
    return experiment


low_rate_data = load_full_cell_low_rate(FULL_CELL_LOW_RATE_FILES)
CELL_CAPACITY_AH = low_rate_data["capacity_ah"]
experimental_curves = load_dynamic_data(DYNAMIC_FILE)

print(f"C/50 charge curves    : {low_rate_data['number_of_charge_curves']}")
print(f"C/50 discharge curves : {low_rate_data['number_of_discharge_curves']}")
print(f"Cell capacity          : {CELL_CAPACITY_AH:.6f} Ah")


## 1. 공통 기반 고정

OCP 비교 전에 두 모델에 동일하게 적용한다.

- Ai2020 electrolyte diffusivity: 원 구현의 단위 해석에 따라 `× 10⁻⁴`
- 음극 평균 입자반경: `3.7542 μm`
- 양극 평균 입자반경: `4.2387 μm`
- 전극 겹침 면적: `45.4 mm × 51.0 mm × 34면`

따라서 뒤의 차이는 입자반경 변경이 아니라 OCP/window 변경에서 발생한다.


In [ ]:
FARADAY_CONSTANT = 96485.33212
NEGATIVE_RADIUS_M = 3.7542e-6
POSITIVE_RADIUS_M = 4.2387e-6
OVERLAP_AREA_M2 = 45.4e-3 * 51.0e-3 * 34.0

raw_parameters = pybamm.ParameterValues("Ai2020")
raw_electrolyte_diffusivity = raw_parameters[
    "Electrolyte diffusivity [m2.s-1]"
]


def corrected_electrolyte_diffusivity(concentration, temperature):
    return 1e-4 * raw_electrolyte_diffusivity(concentration, temperature)


common_parameters = raw_parameters.copy()
common_parameters.update(
    {
        "Electrolyte diffusivity [m2.s-1]": corrected_electrolyte_diffusivity,
        "Negative particle radius [m]": NEGATIVE_RADIUS_M,
        "Positive particle radius [m]": POSITIVE_RADIUS_M,
        "Electrode height [m]": 51.0e-3,
        "Electrode width [m]": 45.4e-3,
        "Number of electrodes connected in parallel to make a cell": 34.0,
    }
)

common_changes = pd.DataFrame(
    [
        ["Electrolyte diffusivity", "Ai2020 function", "Ai2020 function × 1e-4", "unit correction"],
        ["Negative particle radius", f"{1e6 * float(raw_parameters['Negative particle radius [m]']):.4f} μm", "3.7542 μm", "SEM/ImageJ mean"],
        ["Positive particle radius", f"{1e6 * float(raw_parameters['Positive particle radius [m]']):.4f} μm", "4.2387 μm", "SEM/ImageJ mean"],
    ],
    columns=["Parameter", "Ai2020", "Stage-1 common value", "Basis"],
)
display(common_changes)


## 2. OCP와 stoichiometry window

음극은 absolute stoichiometry 축이 정의된 Ai2020 graphite OCP를 유지한다.
양극은 두 GITT 반복시험의 장시간 완화전압을 평균해 사용한다.

Window 최적화 목적함수는 C/50 충·방전 평균 qOCV의 2–98% SOC RMSE이며,
SOC 0/100% 끝점 오차는 각각 10 mV 이내로 제한한다.

양극 OCP 비교 범위는 최적화된 full-cell 3.0–4.2 V 운전 window의
`y100–y0`로 제한한다. 4.2 V는 full-cell 단자전압이므로 양극 OCP를
`4.2 V vs Li/Li+`에서 직접 절단하지 않는다.


In [ ]:
PUBLISHED_WINDOW = {"x0": 0.0065, "x100": 0.8400, "y0": 0.9651, "y100": 0.4350}


def evaluate_parameter_function(parameter_values, name, grid):
    function = parameter_values[name]
    return np.array(
        [
            float(
                np.asarray(
                    function(pybamm.Scalar(float(value))).evaluate(), dtype=float
                ).reshape(-1)[0]
            )
            for value in grid
        ]
    )


ocp_grid = np.linspace(0.001, 0.999, 3001)
AI2020_OCP = {
    "negative": (
        ocp_grid,
        evaluate_parameter_function(common_parameters, "Negative electrode OCP [V]", ocp_grid),
    ),
    "positive": (
        ocp_grid,
        evaluate_parameter_function(common_parameters, "Positive electrode OCP [V]", ocp_grid),
    ),
}


In [ ]:
def duration_to_seconds(value):
    if pd.isna(value):
        return np.nan
    if hasattr(value, "total_seconds"):
        return float(value.total_seconds())
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value) * 86400.0
    parts = str(value).strip().split(":")
    if len(parts) != 3:
        return np.nan
    hours, minutes, seconds = map(float, parts)
    return 3600.0 * hours + 60.0 * minutes + seconds


def duration_seconds(values):
    return values.map(duration_to_seconds)



def build_cathode_gitt_ocp(file_names, parameters):
    disk_area_m2 = np.pi * (0.01 / 2.0) ** 2
    disk_capacity_ah = (
        FARADAY_CONSTANT
        * float(parameters["Maximum concentration in positive electrode [mol.m-3]"])
        * float(parameters["Positive electrode active material volume fraction"])
        * float(parameters["Positive electrode thickness [m]"])
        * disk_area_m2
        / 3600.0
    )
    lithiation_curves, delithiation_curves = [], []

    for file_name in file_names:
        step = pd.read_excel(file_name, sheet_name="step")
        record = pd.read_excel(file_name, sheet_name="record")
        step["duration_s"] = duration_seconds(step["Step Time"])
        cycle = step[step["Cycle Index"] == 2].reset_index(drop=True)
        record["Date"] = pd.to_datetime(record["Date"])
        record = record.sort_values("Date").reset_index(drop=True)
        record_time_ns = record["Date"].to_numpy(dtype="datetime64[ns]").astype(np.int64)
        cumulative = {"CC Chg": 0.0, "CC DChg": 0.0}
        points = {"CC Chg": [], "CC DChg": []}

        for row_index, row in cycle.iterrows():
            step_type = str(row["Step Type"])
            if step_type not in cumulative or not (0 < row["duration_s"] <= 700):
                continue
            rest_index = row_index + 1
            following_rests = []
            while rest_index < len(cycle) and str(cycle.iloc[rest_index]["Step Type"]) == "Rest":
                if cycle.iloc[rest_index]["duration_s"] >= 3500:
                    following_rests.append(cycle.iloc[rest_index])
                rest_index += 1
            if not following_rests:
                continue
            relaxed_state = following_rests[-1]

            onset_ns = pd.Timestamp(row["Oneset Date"]).value
            end_ns = pd.Timestamp(row["End Date"]).value
            left = int(np.searchsorted(record_time_ns, onset_ns, side="left"))
            right = int(np.searchsorted(record_time_ns, end_ns, side="right"))
            pulse_record = record.iloc[left:right]
            pulse_record = pulse_record[pulse_record["Step Type"].astype(str) == step_type]
            pulse_capacity = pd.to_numeric(
                pulse_record["Capacity(Ah)"], errors="coerce"
            ).max()
            cumulative[step_type] += float(pulse_capacity)
            points[step_type].append(
                (cumulative[step_type], float(relaxed_state["End Voltage(V)"]))
            )

        charge = np.asarray(points["CC Chg"], dtype=float)
        discharge = np.asarray(points["CC DChg"], dtype=float)
        delithiation_curves.append(
            clean_curve(1.0 - charge[:, 0] / disk_capacity_ah, charge[:, 1])
        )
        discharge_span = discharge[:, 0].max() / disk_capacity_ah
        lithiation_curves.append(
            clean_curve((1.0 - discharge_span) + discharge[:, 0] / disk_capacity_ah, discharge[:, 1])
        )

    mean_lithiation = mean_overlap(lithiation_curves)
    mean_delithiation = mean_overlap(delithiation_curves)
    return mean_overlap([mean_lithiation, mean_delithiation])




CATHODE_GITT_OCP = build_cathode_gitt_ocp(
    CATHODE_GITT_FILES, raw_parameters
)


In [ ]:
# [3-2] 각 OCP 조합에서 stoichiometry window를 독립적으로 선정한다.
def predict_full_cell_ocv(window_vector, anode_curve, cathode_curve, soc=SOC_GRID):
    x0, x100, y100, y0 = map(float, window_vector)
    negative_stoichiometry = x0 + soc * (x100 - x0)
    positive_stoichiometry = y0 - soc * (y0 - y100)
    negative_ocp = np.interp(negative_stoichiometry, *anode_curve)
    positive_ocp = np.interp(positive_stoichiometry, *cathode_curve)
    return positive_ocp - negative_ocp


def fit_stoichiometry_window(anode_curve, cathode_curve, random_seed):
    anode_min, anode_max = float(anode_curve[0].min()), float(anode_curve[0].max())
    cathode_min, cathode_max = float(cathode_curve[0].min()), float(cathode_curve[0].max())
    minimum_width = CELL_CAPACITY_AH / 6.0
    maximum_width = CELL_CAPACITY_AH / 2.4
    fitting_region = (SOC_GRID >= 0.02) & (SOC_GRID <= 0.98)

    def voltage_mse(window_vector):
        error = (
            predict_full_cell_ocv(window_vector, anode_curve, cathode_curve)
            - low_rate_data["qocv"]
        )
        return float(np.mean(error[fitting_region] ** 2))

    def constraint_violation(window_vector):
        x0, x100, y100, y0 = window_vector
        delta_x, delta_y = x100 - x0, y0 - y100
        endpoint_error = (
            predict_full_cell_ocv(
                window_vector, anode_curve, cathode_curve, SOC_GRID[[0, -1]]
            )
            - low_rate_data["qocv"][[0, -1]]
        )
        violations = [
            max(0.0, minimum_width - delta_x),
            max(0.0, delta_x - maximum_width),
            max(0.0, minimum_width - delta_y),
            max(0.0, delta_y - maximum_width),
            max(0.0, abs(endpoint_error[0]) - 0.010),
            max(0.0, abs(endpoint_error[1]) - 0.010),
        ]
        return float(np.sum(np.square(violations)))

    def penalized_objective(window_vector):
        return voltage_mse(window_vector) + 1e4 * constraint_violation(window_vector)

    bounds = [
        (anode_min, anode_max),
        (anode_min, anode_max),
        (cathode_min, cathode_max),
        (cathode_min, cathode_max),
    ]
    global_result = differential_evolution(
        penalized_objective,
        bounds=bounds,
        seed=random_seed,
        popsize=16,
        maxiter=350,
        tol=1e-10,
        polish=False,
        workers=1,
    )
    constraints = [
        {"type": "ineq", "fun": lambda z: (z[1] - z[0]) - minimum_width},
        {"type": "ineq", "fun": lambda z: maximum_width - (z[1] - z[0])},
        {"type": "ineq", "fun": lambda z: (z[3] - z[2]) - minimum_width},
        {"type": "ineq", "fun": lambda z: maximum_width - (z[3] - z[2])},
        {
            "type": "ineq",
            "fun": lambda z: 0.010
            - abs(
                predict_full_cell_ocv(z, anode_curve, cathode_curve, SOC_GRID[[0]])[0]
                - low_rate_data["qocv"][0]
            ),
        },
        {
            "type": "ineq",
            "fun": lambda z: 0.010
            - abs(
                predict_full_cell_ocv(z, anode_curve, cathode_curve, SOC_GRID[[-1]])[0]
                - low_rate_data["qocv"][-1]
            ),
        },
    ]
    local_result = minimize(
        voltage_mse,
        x0=global_result.x,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": 3000, "ftol": 1e-14, "disp": False},
    )
    if constraint_violation(local_result.x) <= constraint_violation(global_result.x) + 1e-12:
        return np.asarray(local_result.x, dtype=float)
    return np.asarray(global_result.x, dtype=float)




def vector_to_window(vector):
    x0, x100, y100, y0 = map(float, vector)
    return {"x0": x0, "x100": x100, "y0": y0, "y100": y100}


published_vector = np.array([
    PUBLISHED_WINDOW["x0"], PUBLISHED_WINDOW["x100"],
    PUBLISHED_WINDOW["y100"], PUBLISHED_WINDOW["y0"],
])
ai_refit_vector = fit_stoichiometry_window(
    AI2020_OCP["negative"], AI2020_OCP["positive"], random_seed=260924
)
improved_vector = fit_stoichiometry_window(
    AI2020_OCP["negative"], CATHODE_GITT_OCP, random_seed=260921
)
AI_REFIT_WINDOW = vector_to_window(ai_refit_vector)
IMPROVED_WINDOW = vector_to_window(improved_vector)


In [ ]:
# 양극 OCP는 full-cell 3.0–4.2 V에 대응하는 최종 stoichiometry
# window에서 비교한다. 4.2 V는 full-cell 단자전압이므로 양극 OCP를
# 4.2 V vs Li/Li+에서 수평 절단하지 않는다.
y_charge_side = IMPROVED_WINDOW["y100"]
y_discharge_side = IMPROVED_WINDOW["y0"]
comparison_y = np.linspace(y_charge_side, y_discharge_side, 1000)
ai2020_positive_in_window = np.interp(
    comparison_y, *AI2020_OCP["positive"]
)
measured_positive_in_window = np.interp(
    comparison_y, *CATHODE_GITT_OCP
)

figure, axis = plt.subplots(figsize=(8.2, 4.8), constrained_layout=True)
# 범위 밖 곡선은 출처와 측정 범위 확인용으로만 흐리게 표시한다.
axis.plot(
    *AI2020_OCP["positive"], color="#777777", lw=1.2, alpha=0.25
)
axis.plot(
    *CATHODE_GITT_OCP, color="#D55E00", lw=1.2, alpha=0.25
)
axis.axvspan(
    y_charge_side,
    y_discharge_side,
    color="#56B4E9",
    alpha=0.09,
    label="Full-cell 3.0–4.2 V stoichiometry window",
)
axis.plot(
    comparison_y,
    ai2020_positive_in_window,
    color="#666666",
    lw=2.3,
    label="Ai2020 cathode OCP",
)
axis.plot(
    comparison_y,
    measured_positive_in_window,
    color="#D55E00",
    lw=2.5,
    label="Measured cathode GITT OCP",
)
axis.axvline(
    y_charge_side,
    color="#CC3311",
    ls="--",
    lw=1.5,
    label=f"Full-cell 4.2 V side: y100={y_charge_side:.4f}",
)
axis.axvline(
    y_discharge_side,
    color="#0072B2",
    ls="--",
    lw=1.5,
    label=f"Full-cell 3.0 V side: y0={y_discharge_side:.4f}",
)
margin = 0.03
axis.set_xlim(
    max(float(CATHODE_GITT_OCP[0].min()), y_charge_side - margin),
    min(float(CATHODE_GITT_OCP[0].max()), y_discharge_side + margin),
)
axis.set(
    xlabel="Cathode stoichiometry y",
    ylabel="OCP vs Li/Li+ [V]",
    title="Cathode OCP over the full-cell operating window",
)
axis.grid(alpha=0.25)
axis.legend(fontsize=8)
plt.show()

# qOCV: 공개 window → window만 재최적화 → 실험 양극 OCP+window 재최적화
qocv_cases = [
    ("Ai2020 published", published_vector, AI2020_OCP["negative"], AI2020_OCP["positive"], "#999999", "--"),
    ("Ai2020 OCP + refit window", ai_refit_vector, AI2020_OCP["negative"], AI2020_OCP["positive"], "#0072B2", "-"),
    ("Measured cathode OCP + refit window", improved_vector, AI2020_OCP["negative"], CATHODE_GITT_OCP, "#D55E00", "-"),
]
fitting_region = (SOC_GRID >= 0.02) & (SOC_GRID <= 0.98)
qocv_rows, qocv_predictions = [], {}

for name, vector, anode_ocp, cathode_ocp, color, style in qocv_cases:
    prediction = predict_full_cell_ocv(vector, anode_ocp, cathode_ocp)
    qocv_predictions[name] = prediction
    error_mv = 1000.0 * (prediction - low_rate_data["qocv"])
    x0, x100, y100, y0 = vector
    qocv_rows.append(
        {
            "Case": name,
            "x0": x0, "x100": x100, "y0": y0, "y100": y100,
            "Qn [Ah]": CELL_CAPACITY_AH / (x100 - x0),
            "Qp [Ah]": CELL_CAPACITY_AH / (y0 - y100),
            "qOCV RMSE 2–98% [mV]": np.sqrt(np.mean(error_mv[fitting_region] ** 2)),
            "SOC0 error [mV]": error_mv[0],
            "SOC100 error [mV]": error_mv[-1],
        }
    )

window_results = pd.DataFrame(qocv_rows)
display(window_results.round(6))

figure, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
axes[0].plot(SOC_GRID * 100, low_rate_data["qocv"], color="black", lw=2.7, label="Measured C/50 qOCV")
axes[1].axhline(0.0, color="black", lw=0.8)
for name, _, _, _, color, style in qocv_cases:
    prediction = qocv_predictions[name]
    axes[0].plot(SOC_GRID * 100, prediction, color=color, ls=style, lw=2.0, label=name)
    axes[1].plot(SOC_GRID * 100, 1000 * (prediction - low_rate_data["qocv"]), color=color, ls=style, lw=1.8, label=name)
axes[0].set(title="qOCV reconstruction", xlabel="Full-cell SOC [%]", ylabel="Voltage [V]")
axes[1].set(title="qOCV error", xlabel="Full-cell SOC [%]", ylabel="Model − experiment [mV]")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
plt.show()


## 3. Capacity-consistent inventory와 DFN 검증

개선 window에서 계산된 `Q_n`, `Q_p`를 같은 전극 면적·두께·활물질 분율로 구현하도록
`c_s,max,n/p`를 역산한다. 이 값은 독립 fitting 변수가 아니라 window와 용량을 연결하는 bookkeeping이다.


In [ ]:
def build_improved_parameters(base_parameters, window, positive_ocp):
    parameters = base_parameters.copy()
    qn_ah = CELL_CAPACITY_AH / (window["x100"] - window["x0"])
    qp_ah = CELL_CAPACITY_AH / (window["y0"] - window["y100"])
    csmax_n = 3600.0 * qn_ah / (
        FARADAY_CONSTANT * OVERLAP_AREA_M2
        * float(parameters["Negative electrode thickness [m]"])
        * float(parameters["Negative electrode active material volume fraction"])
    )
    csmax_p = 3600.0 * qp_ah / (
        FARADAY_CONSTANT * OVERLAP_AREA_M2
        * float(parameters["Positive electrode thickness [m]"])
        * float(parameters["Positive electrode active material volume fraction"])
    )
    positive_grid, positive_voltage = positive_ocp

    def measured_positive_ocp(stoichiometry):
        return pybamm.Interpolant(
            positive_grid, positive_voltage, stoichiometry,
            name="Measured cathode GITT OCP", interpolator="linear"
        )

    parameters.update(
        {
            "Maximum concentration in negative electrode [mol.m-3]": csmax_n,
            "Maximum concentration in positive electrode [mol.m-3]": csmax_p,
            "Positive electrode OCP [V]": measured_positive_ocp,
        }
    )
    return parameters, csmax_n, csmax_p


improved_parameters, improved_csmax_n, improved_csmax_p = build_improved_parameters(
    common_parameters, IMPROVED_WINDOW, CATHODE_GITT_OCP
)
baseline_csmax_n = float(common_parameters["Maximum concentration in negative electrode [mol.m-3]"])
baseline_csmax_p = float(common_parameters["Maximum concentration in positive electrode [mol.m-3]"])

parameter_summary = pd.DataFrame(
    [
        ["Negative OCP", "Ai2020", "Ai2020", "absolute x-axis retained"],
        ["Positive OCP", "Ai2020", "Measured GITT mean", "cell-specific relaxed voltage"],
        ["x0 / x100", f"{PUBLISHED_WINDOW['x0']:.6f} / {PUBLISHED_WINDOW['x100']:.6f}", f"{IMPROVED_WINDOW['x0']:.6f} / {IMPROVED_WINDOW['x100']:.6f}", "C/50 qOCV fit"],
        ["y0 / y100", f"{PUBLISHED_WINDOW['y0']:.6f} / {PUBLISHED_WINDOW['y100']:.6f}", f"{IMPROVED_WINDOW['y0']:.6f} / {IMPROVED_WINDOW['y100']:.6f}", "C/50 qOCV fit"],
        ["c_s,max,n [mol m-3]", f"{baseline_csmax_n:.1f}", f"{improved_csmax_n:.1f}", "Qn/window consistency"],
        ["c_s,max,p [mol m-3]", f"{baseline_csmax_p:.1f}", f"{improved_csmax_p:.1f}", "Qp/window consistency"],
    ],
    columns=["Item", "Baseline", "Improved", "Basis"],
)
display(parameter_summary)


In [ ]:
# [결과-1] DFN 실행 및 실험 대비 오차 계산
def stage_ocv(stage):
    window = stage["window"]
    return predict_full_cell_ocv(
        [window["x0"], window["x100"], window["y100"], window["y0"]],
        stage["anode_ocp"],
        stage["cathode_ocp"],
    )


def initial_soc_from_rest_voltage(stage, rest_voltage_v):
    qocv = stage_ocv(stage)
    order = np.argsort(qocv, kind="stable")
    unique_voltage, unique_index = np.unique(qocv[order], return_index=True)
    return float(np.interp(rest_voltage_v, unique_voltage, SOC_GRID[order][unique_index]))


def run_dfn_branch(stage, c_rate, is_charge):
    experiment = experimental_curves[(c_rate, is_charge)]
    initial_soc = initial_soc_from_rest_voltage(stage, experiment["rest_voltage_v"])
    window = stage["window"]
    initial_x = window["x0"] + initial_soc * (window["x100"] - window["x0"])
    initial_y = window["y0"] - initial_soc * (window["y0"] - window["y100"])
    branch_parameters = stage["parameters"].copy()
    branch_parameters.update(
        {
            "Initial concentration in negative electrode [mol.m-3]": initial_x * stage["c_s_max_n"],
            "Initial concentration in positive electrode [mol.m-3]": initial_y * stage["c_s_max_p"],
        }
    )

    direction = "Charge" if is_charge else "Discharge"
    cutoff_voltage = 4.2 if is_charge else 3.0
    protocol = f"{direction} at {experiment['current_a']} A until {cutoff_voltage} V"
    simulation = pybamm.Simulation(
        pybamm.lithium_ion.DFN({"thermal": "isothermal"}),
        parameter_values=branch_parameters,
        experiment=pybamm.Experiment([protocol], period="10 seconds"),
        var_pts={"x_n": 15, "x_s": 15, "x_p": 15, "r_n": 15, "r_p": 15},
        solver=pybamm.IDAKLUSolver(rtol=1e-6, atol=1e-8),
    )
    solution = simulation.solve()
    time_s = np.asarray(solution["Time [s]"].entries, dtype=float)
    voltage_v = np.asarray(solution["Terminal voltage [V]"].entries, dtype=float)
    current_a = np.asarray(solution["Current [A]"].entries, dtype=float)
    anode_potential_v = np.asarray(
        solution["Negative electrode surface potential difference at separator interface [V]"].entries,
        dtype=float,
    ).squeeze()
    capacity_ah = cumulative_trapezoid(np.abs(current_a), time_s, initial=0.0) / 3600.0
    if is_charge:
        full_cell_soc_pct = 100.0 * (initial_soc + capacity_ah / CELL_CAPACITY_AH)
    else:
        full_cell_soc_pct = 100.0 * (initial_soc - capacity_ah / CELL_CAPACITY_AH)
    return {
        "time_min": time_s / 60.0,
        "voltage_v": voltage_v,
        "anode_potential_v": anode_potential_v,
        "capacity_ah": capacity_ah,
        "full_cell_soc_pct": np.clip(full_cell_soc_pct, 0.0, 100.0),
        "initial_soc_pct": 100.0 * initial_soc,
    }


def compare_with_experiment(experiment, simulation):
    experiment_time = experiment["time_min"] - experiment["time_min"][0]
    simulation_time = simulation["time_min"] - simulation["time_min"][0]
    common_end = 0.995 * min(experiment_time[-1], simulation_time[-1])
    full_grid = np.linspace(0.0, common_end, 400)
    center_start = 0.10 * CELL_CAPACITY_AH / experiment["current_a"] * 60.0
    center_end = min(0.70 * CELL_CAPACITY_AH / experiment["current_a"] * 60.0, common_end)
    if center_end <= center_start:
        center_grid = full_grid[full_grid >= 0.10 * common_end]
    else:
        center_grid = np.linspace(center_start, center_end, 160)

    def error_mv(time_grid):
        return 1000.0 * (
            np.interp(time_grid, simulation_time, simulation["voltage_v"])
            - np.interp(time_grid, experiment_time, experiment["voltage_v"])
        )

    full_error = error_mv(full_grid)
    center_error = error_mv(center_grid)
    experiment_capacity = experiment["capacity_ah"][-1]
    simulation_capacity = simulation["capacity_ah"][-1]
    return {
        "full_RMSE_mV": float(np.sqrt(np.mean(full_error**2))),
        "full_MAE_mV": float(np.mean(np.abs(full_error))),
        "center_10_70_RMSE_mV": float(np.sqrt(np.mean(center_error**2))),
        "center_10_70_MAE_mV": float(np.mean(np.abs(center_error))),
        "capacity_error_pct": float(100.0 * (simulation_capacity - experiment_capacity) / experiment_capacity),
    }


In [ ]:
stages = {
    "Baseline: Ai2020 OCP/window": {
        "parameters": common_parameters,
        "window": PUBLISHED_WINDOW,
        "c_s_max_n": baseline_csmax_n,
        "c_s_max_p": baseline_csmax_p,
        "anode_ocp": AI2020_OCP["negative"],
        "cathode_ocp": AI2020_OCP["positive"],
    },
    "Improved: measured cathode OCP/refit window": {
        "parameters": improved_parameters,
        "window": IMPROVED_WINDOW,
        "c_s_max_n": improved_csmax_n,
        "c_s_max_p": improved_csmax_p,
        "anode_ocp": AI2020_OCP["negative"],
        "cathode_ocp": CATHODE_GITT_OCP,
    },
}

simulations, metric_rows = {}, []
for stage_name, stage in stages.items():
    for c_rate in TEST_C_RATES:
        for is_charge in (True, False):
            direction = "Charge" if is_charge else "Discharge"
            print(f"{stage_name} | {c_rate:g}C {direction}")
            simulation = run_dfn_branch(stage, c_rate, is_charge)
            simulations[(stage_name, c_rate, is_charge)] = simulation
            metric_rows.append(
                {
                    "Model": stage_name,
                    "C-rate": c_rate,
                    "Direction": direction,
                    **compare_with_experiment(
                        experimental_curves[(c_rate, is_charge)], simulation
                    ),
                }
            )

metrics = pd.DataFrame(metric_rows)
display(metrics.round(3))


In [ ]:
def rms(values):
    values = np.asarray(values, dtype=float)
    return float(np.sqrt(np.mean(values**2)))


model_summary = (
    metrics.groupby("Model", as_index=False)
    .agg(
        Voltage_full_RMSE_mV=("full_RMSE_mV", rms),
        Voltage_10_70_RMSE_mV=("center_10_70_RMSE_mV", rms),
        Cutoff_capacity_RMSE_pct=("capacity_error_pct", rms),
        Max_abs_cutoff_capacity_error_pct=("capacity_error_pct", lambda x: float(np.max(np.abs(x)))),
    )
)
display(model_summary.round(3))

colors = {
    "Baseline: Ai2020 OCP/window": "#888888",
    "Improved: measured cathode OCP/refit window": "#D55E00",
}
figure, axes = plt.subplots(2, 3, figsize=(15.5, 8.0), sharey=True, constrained_layout=True)
for column, c_rate in enumerate(TEST_C_RATES):
    for row, is_charge in enumerate((True, False)):
        direction = "Charge" if is_charge else "Discharge"
        axis = axes[row, column]
        experiment = experimental_curves[(c_rate, is_charge)]
        axis.plot(experiment["time_min"], experiment["voltage_v"], color="black", lw=2.7, label="Experiment")
        for stage_name in stages:
            simulation = simulations[(stage_name, c_rate, is_charge)]
            branch = metrics[
                (metrics["Model"] == stage_name)
                & np.isclose(metrics["C-rate"], c_rate)
                & (metrics["Direction"] == direction)
            ].iloc[0]
            axis.plot(
                simulation["time_min"], simulation["voltage_v"],
                color=colors[stage_name], lw=2.0,
                ls="--" if stage_name.startswith("Baseline") else "-",
                label=f"{stage_name.split(':')[0]} ({branch.full_RMSE_mV:.1f} mV)",
            )
        axis.set_title(f"{c_rate:g}C {direction}", weight="bold")
        axis.set_ylim(2.9, 4.3)
        axis.grid(alpha=0.25)
        if row == 1:
            axis.set_xlabel("Elapsed time [min]")
        if column == 0:
            axis.set_ylabel("Cell voltage [V]")
handles, labels = axes[0, 0].get_legend_handles_labels()
figure.legend(handles, labels, loc="outside upper center", ncol=3, frameon=False)
figure.suptitle("Stage 1: baseline vs OCP/window improvement", weight="bold")
plt.show()


## 해석 원칙

- qOCV 표는 **window 재최적화 효과**와 **양극 OCP 교체의 추가 효과**를 분리해 보여준다.
- 동적 표의 전압 오차는 실험·모델 공통 시간 구간의 RMSE이다.
- `cut-off capacity error`는 4.2/3.0 V 도달 용량의 상대오차이며, 독립적인 용량 예측 오차로 보고한다.
- 개선 모델에서 양극 표면 stoichiometry가 GITT OCP 상한을 미세하게 넘으면 PyBaMM 경고가 발생할 수 있다. 경고를 숨기지 않고 OCP 측정 범위의 한계로 남긴다.
- 이 단계에서 `D_s`, `k`, Bruggeman 계수는 변경하지 않았으므로 이후 단계의 fitting 효과와 섞이지 않는다.
